# Anonymous Support and Advice Portal
**Say it to the sea. Someone will answer.**

Programming Lab - Assignment 3

**Group members:**
- Zamar Murtaza (25L-5083)
- Haania Arif (25L-5193)

## About our portal
Sometimes students are stressed, lonely or confused but they don't want to tell someone they know. In our portal a person can ask for help without showing their name, like a message in a bottle.

- A user writes a message (a "bottle") and says if they need **advice**, **comfort** or **motivation**.
- The bottle drifts for a little while, then another user can catch it at random.
- That user can reply to it or throw it back into the sea.
- The person who threw the bottle can mark a reply as helpful. People with many helpful replies become **Lighthouses**.
- Bad replies can be reported.
- Nobody sees real usernames. Everyone gets a random name like "Silver Anchor 42".

## Python concepts we used
- **Functions** for every feature, with parameters and return values
- **while True** loops for the menus and **for** loops to search and count data
- **Lists, dictionaries and tuples** to store the data
- **File handling**: all data is saved in a JSON file, and every action is added to a log file
- **try / except / finally** so the program does not crash on wrong input or file problems

## Part 1: The code

### 1. Imports and settings
We import `json` to save our data in a file and `random` to make random names and pick random bottles.

The bottle types, moods and name parts are **tuples** because they never change. The numbers at the bottom are the rules of the portal.

In [1]:
import json
import random

DATA_FILE = "portal_data.json"
LOG_FILE = "activity_log.txt"
REPORT_FILE = "portal_report.txt"

BOTTLE_TYPES = ("advice", "comfort", "motivation")
MOODS = ("sad", "stressed", "confused", "lost", "anxious", "lonely", "other")
FIRST_NAMES = ("Quiet", "Lost", "Silver", "Gentle", "Calm", "Hidden", "Blue")
SECOND_NAMES = ("Wave", "Anchor", "Shell", "Tide", "Pearl", "Gull")
DANGER_WORDS = ("suicide", "kill myself", "end my life", "want to die", "self harm")

CATCH_LIMIT = 3       # how many bottles a user can catch in one login
LIGHTHOUSE_LIMIT = 3  # helpful replies needed to become a Lighthouse

### 2. File handling
All our data is kept in one dictionary. It has a list of users, a list of bottles, a list of replies and a list of reports. Every user, bottle and reply is a small dictionary inside these lists.

- `load_data()` opens the JSON file when the portal starts and reads the data. If the file is not there (first time), we catch `FileNotFoundError` and add some demo data. If the file is broken we catch `json.JSONDecodeError`. `finally` closes the file in every case. At the end a for loop checks that no part of the data is missing.
- `save_data()` opens the file in `"w"` mode and writes all the data again.
- `write_log()` opens the log file in `"a"` (append) mode, so a new line is added at the end and old lines stay.

In [2]:
def load_data(filename):
    """Reads all portal data from the JSON file. Returns the data dictionary."""
    data = {"tide": 0, "users": [], "bottles": [], "replies": [], "reports": []}
    file = None
    try:
        file = open(filename, "r")
        data = json.load(file)
        print("Data loaded from", filename)
    except FileNotFoundError:
        print("No data file found. Starting fresh with some demo data.")
        add_demo_data(data)
    except PermissionError:
        print("Error: not allowed to open", filename)
    except json.JSONDecodeError:
        print("Error: the data file is damaged. Starting with empty data.")
    finally:
        if file != None:
            file.close()

    # make sure no part of the data is missing
    for key in ["users", "bottles", "replies", "reports"]:
        if key not in data:
            data[key] = []
    if "tide" not in data:
        data["tide"] = 0
    return data


def save_data(data, filename):
    """Writes all portal data to the JSON file. Returns True if it worked."""
    file = None
    try:
        file = open(filename, "w")
        json.dump(data, file, indent=4)
        return True
    except PermissionError:
        print("Error: not allowed to save to", filename)
        return False
    finally:
        if file != None:
            file.close()


def write_log(message):
    """Adds one line to the activity log (append mode). Returns True if it worked."""
    try:
        file = open(LOG_FILE, "a")
        file.write(message + "\n")
        file.close()
        return True
    except PermissionError:
        print("Error: could not write to the log file")
        return False

### 3. Taking numbers from the user safely
If the user types letters when we ask for a number, `int()` gives a `ValueError`. We catch it so the program does not crash.

- `check_number()` tries to change the text into a number and checks it is in the right range. It returns the number, or `None` if it is wrong.
- `get_number()` keeps asking in a `while True` loop until `check_number()` gives back a correct number.
- `choose_from()` prints a numbered list with a for loop and returns the option the user picks.

In [3]:
def check_number(text, low, high):
    """Changes text into a number between low and high. Returns the number, or None if it is not valid."""
    try:
        number = int(text)
    except ValueError:
        print("Please enter a number, not letters.")
        return None
    if number < low or number > high:
        print("Please enter a number from", low, "to", high)
        return None
    return number


def get_number(question, low, high):
    """Keeps asking until the user types a valid number. Returns the number."""
    while True:
        text = input(question)
        number = check_number(text, low, high)
        if number != None:
            return number


def choose_from(options, title):
    """Shows a numbered list of options. Returns the option the user picks."""
    print(title)
    for i in range(len(options)):
        print(f"  {i + 1}. {options[i]}")
    choice = get_number("Your choice: ", 1, len(options))
    return options[choice - 1]

### 4. Small helper functions
These are small functions we use many times in the program.

- `find_user()` and `find_by_id()` go through a list with a for loop and return the matching item, or `None`.
- `make_alias()` makes a random name from the two tuples. The while loop keeps trying until it finds a name nobody else has.
- `has_danger_words()` checks if a message has words that show the person might be in danger.

In [4]:
def find_user(data, username):
    """Finds a user by username. Returns the user, or None."""
    for user in data["users"]:
        if user["username"] == username:
            return user
    return None


def find_by_id(items, item_id):
    """Finds a user, bottle or reply by its id. Returns the item, or None."""
    for item in items:
        if item["id"] == item_id:
            return item
    return None


def make_alias(data):
    """Makes a random anonymous name that nobody else has. Returns the name."""
    while True:
        alias = random.choice(FIRST_NAMES) + " " + random.choice(SECOND_NAMES) + " " + str(random.randint(10, 99))
        taken = False
        for user in data["users"]:
            if user["alias"] == alias:
                taken = True
        if taken == False:
            return alias


def has_danger_words(message):
    """Checks if a message has any danger words. Returns True or False."""
    for word in DANGER_WORDS:
        if word in message.lower():
            return True
    return False

### 5. Register and login
`register_user()` first checks the username and password with if statements. If everything is fine, it makes a dictionary for the new user, gives them a random alias and adds it to the users list.

`user_login()` finds the user and checks the password. It returns the user if it is correct, or `None` if it is wrong.

(We save passwords as normal text to keep it simple. A real website would hide them.)

In [5]:
def register_user(data, username, password):
    """Creates a new account. Returns True if the account was made."""
    if username == "" or " " in username:
        print("Username cannot be empty or have spaces.")
        return False
    if len(password) < 4:
        print("Password must be at least 4 characters.")
        return False
    if find_user(data, username) != None:
        print("This username is already taken.")
        return False

    user = {
        "id": len(data["users"]) + 1,
        "username": username,
        "password": password,
        "alias": make_alias(data),
        "helped": 0,
    }
    data["users"].append(user)
    write_log("New user joined as " + user["alias"])
    print("Account created! Other people will only see you as:", user["alias"])
    return True


def user_login(data, username, password):
    """Checks the username and password. Returns the user if correct, otherwise None."""
    user = find_user(data, username)
    if user == None or user["password"] != password:
        print("Wrong username or password.")
        return None
    write_log(user["alias"] + " logged in")
    print("Welcome back,", user["alias"])
    return user

### 6. Throwing and catching bottles
This is the main part of our portal.

**Drifting:** we have a counter called `tide` that goes up by 1 every time a user does something in the menu. When a bottle is thrown we set `ready_at` to the current tide plus a random number from 1 to 3. A bottle can only be caught when the tide reaches `ready_at`, so it "drifts" for a while first.

- `throw_bottle()` checks the input and adds a new bottle dictionary to the list (this is our add record function). If the message has danger words, it shows a message telling the person to get help.
- `find_catchable()` uses a for loop with nested ifs. A bottle can be caught only if nobody replied yet, it is not your own bottle, it finished drifting, and you did not catch it before.
- `catch_bottle()` picks one of these bottles with `random.choice()`.
- `reply_bottle()` saves a reply and changes the bottle's status to `"replied"`.
- `throw_back()` gives the bottle a new drift time so someone else can find it.

In [6]:
def throw_bottle(data, user, bottle_type, mood, message):
    """Adds a new bottle to the sea (our add record function). Returns True if it was added."""
    if bottle_type not in BOTTLE_TYPES or mood not in MOODS:
        print("Wrong bottle type or mood.")
        return False
    if len(message) < 10:
        print("Message is too short, please write a bit more.")
        return False

    bottle = {
        "id": len(data["bottles"]) + 1,
        "sender_id": user["id"],
        "type": bottle_type,
        "mood": mood,
        "message": message,
        "status": "drifting",
        "ready_at": data["tide"] + random.randint(1, 3),
        "touched": 0,
        "caught_by": [],
    }
    data["bottles"].append(bottle)
    write_log(user["alias"] + " threw bottle " + str(bottle["id"]))
    print("Your bottle is drifting out to sea...")

    if has_danger_words(message):
        print("It sounds like you are going through something very hard.")
        print("Please talk to someone you trust or call a mental health helpline.")
    return True


def find_catchable(data, user):
    """Returns a list of bottles that this user is allowed to catch."""
    catchable = []
    for bottle in data["bottles"]:
        if bottle["status"] == "drifting":
            if bottle["sender_id"] != user["id"]:
                if bottle["ready_at"] <= data["tide"]:
                    if user["id"] not in bottle["caught_by"]:
                        catchable.append(bottle)
    return catchable


def catch_bottle(data, user):
    """Picks a random bottle from the shore. Returns the bottle, or None if there are none."""
    catchable = find_catchable(data, user)
    if len(catchable) == 0:
        return None
    bottle = random.choice(catchable)
    bottle["touched"] = bottle["touched"] + 1
    bottle["caught_by"].append(user["id"])
    write_log(user["alias"] + " caught bottle " + str(bottle["id"]))
    return bottle


def reply_bottle(data, user, bottle, text):
    """Saves a reply to a bottle. Returns True if the reply was saved."""
    if bottle["status"] != "drifting":
        print("Someone already replied to this bottle.")
        return False
    if len(text) < 10:
        print("Reply is too short, please write a bit more.")
        return False

    reply = {
        "id": len(data["replies"]) + 1,
        "bottle_id": bottle["id"],
        "replier_id": user["id"],
        "message": text,
        "helpful": False,
    }
    data["replies"].append(reply)
    bottle["status"] = "replied"
    write_log(user["alias"] + " replied to bottle " + str(bottle["id"]))
    print("Your reply is floating back to them. Thank you for being kind!")
    return True


def throw_back(data, bottle):
    """Lets a bottle drift again so someone else can catch it. Returns its new ready time."""
    bottle["ready_at"] = data["tide"] + random.randint(1, 3)
    write_log("Bottle " + str(bottle["id"]) + " was thrown back")
    print("You let the bottle drift away.")
    return bottle["ready_at"]

### 7. My bottles, helpful replies and reports
- `get_my_bottles()` has a for loop inside another for loop. The outer loop finds the user's bottles and the inner loop finds the replies of each bottle. Each result is a tuple `(bottle, replies)`.
- `mark_helpful()` checks that the reply is on the user's own bottle and was not marked before. Then it adds 1 to the helper's count. At 3 helpful replies they become a Lighthouse.
- `report_reply()` saves a report with the reason.

In [7]:
def get_my_bottles(data, user):
    """Returns a list of the user's bottles. Each item is a tuple (bottle, replies)."""
    my_bottles = []
    for bottle in data["bottles"]:
        if bottle["sender_id"] == user["id"]:
            replies = []
            for reply in data["replies"]:
                if reply["bottle_id"] == bottle["id"]:
                    replies.append(reply)
            my_bottles.append((bottle, replies))
    return my_bottles


def mark_helpful(data, user, reply_id):
    """Marks a reply on the user's own bottle as helpful. Returns True if it worked."""
    reply = find_by_id(data["replies"], reply_id)
    if reply == None:
        print("There is no reply with that number.")
        return False
    bottle = find_by_id(data["bottles"], reply["bottle_id"])
    if bottle["sender_id"] != user["id"]:
        print("You can only rate replies to your own bottles.")
        return False
    if reply["helpful"] == True:
        print("You already marked this reply as helpful.")
        return False

    reply["helpful"] = True
    helper = find_by_id(data["users"], reply["replier_id"])
    helper["helped"] = helper["helped"] + 1
    write_log(helper["alias"] + "'s reply was marked helpful")
    print("Thanks! We told", helper["alias"], "that their reply helped.")
    if helper["helped"] == LIGHTHOUSE_LIMIT:
        print(helper["alias"], "is now a Lighthouse!")
    return True


def report_reply(data, user, reply_id, reason):
    """Reports a bad reply. Returns True if the report was saved."""
    if find_by_id(data["replies"], reply_id) == None:
        print("There is no reply with that number.")
        return False
    if reason == "":
        print("Please write a reason.")
        return False

    report = {
        "id": len(data["reports"]) + 1,
        "reply_id": reply_id,
        "reporter_id": user["id"],
        "reason": reason,
    }
    data["reports"].append(report)
    write_log("Reply " + str(reply_id) + " was reported")
    print("Thank you, the reply has been reported.")
    return True

### 8. Search, leaderboard and report
- `search_bottles()` is our search record function. It goes through all bottles and keeps the ones that match the keyword, type and mood. If a filter is empty (`""`) it means "any".
- `get_leaderboard()` makes a list of users who helped someone and sorts it with **bubble sort**. Two nested for loops compare two users next to each other and swap them if they are in the wrong order.
- `generate_report()` counts things using dictionaries (like how many bottles of each type), finds the most common mood, works out the percentage of answered bottles, writes everything into a text file and returns the report.

In [8]:
def search_bottles(data, keyword, bottle_type, mood):
    """Finds bottles matching the keyword, type and mood (our search record function).
    An empty value means any. Returns a list of matching bottles."""
    results = []
    for bottle in data["bottles"]:
        match = True
        if keyword != "" and keyword.lower() not in bottle["message"].lower():
            match = False
        if bottle_type != "" and bottle["type"] != bottle_type:
            match = False
        if mood != "" and bottle["mood"] != mood:
            match = False
        if match == True:
            results.append(bottle)
    return results


def get_leaderboard(data):
    """Returns users who helped someone, sorted from most to least helpful."""
    helpers = []
    for user in data["users"]:
        if user["helped"] > 0:
            helpers.append(user)

    # bubble sort
    for i in range(len(helpers)):
        for j in range(len(helpers) - 1 - i):
            if helpers[j]["helped"] < helpers[j + 1]["helped"]:
                temp = helpers[j]
                helpers[j] = helpers[j + 1]
                helpers[j + 1] = temp
    return helpers


def generate_report(data):
    """Counts statistics about the portal, saves them in a text file and returns the report."""
    type_count = {}
    for bottle_type in BOTTLE_TYPES:
        type_count[bottle_type] = 0
    mood_count = {}
    for mood in MOODS:
        mood_count[mood] = 0

    replied = 0
    for bottle in data["bottles"]:
        type_count[bottle["type"]] = type_count[bottle["type"]] + 1
        mood_count[bottle["mood"]] = mood_count[bottle["mood"]] + 1
        if bottle["status"] == "replied":
            replied = replied + 1

    helpful = 0
    for reply in data["replies"]:
        if reply["helpful"] == True:
            helpful = helpful + 1

    top_mood = "none"
    top_number = 0
    for mood in mood_count:
        if mood_count[mood] > top_number:
            top_mood = mood
            top_number = mood_count[mood]

    total = len(data["bottles"])
    if total > 0:
        percent = round(replied / total * 100)
    else:
        percent = 0

    report = "========== PORTAL REPORT ==========\n"
    report = report + f"Users: {len(data['users'])}\n"
    report = report + f"Bottles thrown: {total}\n"
    report = report + f"Bottles answered: {replied} ({percent}%)\n"
    report = report + f"Replies written: {len(data['replies'])}\n"
    report = report + f"Helpful replies: {helpful}\n"
    report = report + f"Reported replies: {len(data['reports'])}\n"
    report = report + "Bottles by type:\n"
    for bottle_type in type_count:
        report = report + f"   {bottle_type}: {type_count[bottle_type]}\n"
    report = report + f"Most common mood: {top_mood}\n"
    report = report + "Top helpers:\n"
    leaderboard = get_leaderboard(data)
    if len(leaderboard) == 0:
        report = report + "   nobody yet\n"
    count = 0
    for user in leaderboard:
        if count < 3:
            report = report + f"   {count + 1}. {user['alias']} ({user['helped']} helpful)\n"
            count = count + 1

    file = None
    try:
        file = open(REPORT_FILE, "w")
        file.write(report)
    except PermissionError:
        print("Error: could not save the report file")
    finally:
        if file != None:
            file.close()
    return report

### 9. Demo data
When the portal runs for the first time there is no data file, so `load_data()` calls this function. It makes two demo users (**ayesha** and **bilal**, password **1234**) and three bottles so there is something to catch. It uses our `register_user()` and `throw_bottle()` functions again.

In [9]:
def add_demo_data(data):
    """Adds two demo users and three bottles. Returns the number of bottles added."""
    register_user(data, "ayesha", "1234")
    register_user(data, "bilal", "1234")
    ayesha = find_user(data, "ayesha")
    bilal = find_user(data, "bilal")

    throw_bottle(data, ayesha, "motivation", "stressed", "Exams start next week and I feel like I already failed.")
    throw_bottle(data, bilal, "comfort", "lonely", "I moved to a new city for uni and I have no friends here yet.")
    throw_bottle(data, ayesha, "advice", "confused", "Should I tell my parents I want to change my degree?")

    # demo bottles can be caught straight away
    for bottle in data["bottles"]:
        bottle["ready_at"] = 0
    return len(data["bottles"])

### 10. Menus
These functions show the menus, take input from the user and call the functions above.

- `throw_menu()`, `catch_menu()`, `my_bottles_menu()`, `search_menu()` and `leaderboard_menu()` are for each option.
- `user_menu()` is the menu after login. It is a `while True` loop that runs until the user picks Logout. After every action the tide goes up by 1 and the data is saved to the file.

In [10]:
def throw_menu(data, user):
    """Asks the user for a bottle and throws it."""
    bottle_type = choose_from(BOTTLE_TYPES, "What do you need?")
    mood = choose_from(MOODS, "How are you feeling?")
    message = input("Write your message: ").strip()
    throw_bottle(data, user, bottle_type, mood, message)


def catch_menu(data, user, catches_left):
    """Catches a bottle and lets the user reply or throw it back. Returns catches left."""
    if catches_left == 0:
        print("You caught enough bottles for now. Come back later.")
        return 0
    bottle = catch_bottle(data, user)
    if bottle == None:
        print("The shore is empty right now. No bottles have washed up.")
        return catches_left

    print("A bottle washed up at your feet!")
    print("Needs:", bottle["type"], "| Mood:", bottle["mood"])
    print("Message:", bottle["message"])
    if bottle["touched"] > 1:
        print(bottle["touched"] - 1, "other people found this bottle before you.")

    print("1. Reply")
    print("2. Throw it back")
    choice = get_number("Choose: ", 1, 2)
    if choice == 1:
        text = input("Your reply: ").strip()
        saved = reply_bottle(data, user, bottle, text)
        if saved == False:
            throw_back(data, bottle)
    else:
        throw_back(data, bottle)
    return catches_left - 1


def my_bottles_menu(data, user):
    """Shows the user's bottles and replies, and lets them mark helpful or report."""
    my_bottles = get_my_bottles(data, user)
    if len(my_bottles) == 0:
        print("You have not thrown any bottles yet.")
        return

    reply_count = 0
    for bottle, replies in my_bottles:
        print()
        print(f"Bottle {bottle['id']} ({bottle['type']}, {bottle['mood']}) - {bottle['status']}")
        print("  ", bottle["message"])
        for reply in replies:
            helper = find_by_id(data["users"], reply["replier_id"])
            print(f"   Reply {reply['id']} from {helper['alias']}: {reply['message']}")
            if reply["helpful"] == True:
                print("   (you marked this as helpful)")
            reply_count = reply_count + 1

    if reply_count == 0:
        print("No replies yet, your bottles are still drifting.")
        return

    print()
    print("1. Mark a reply as helpful")
    print("2. Report a reply")
    print("3. Go back")
    choice = get_number("Choose: ", 1, 3)
    if choice == 1:
        reply_id = get_number("Reply number: ", 1, len(data["replies"]))
        mark_helpful(data, user, reply_id)
    elif choice == 2:
        reply_id = get_number("Reply number: ", 1, len(data["replies"]))
        reason = input("Why are you reporting it? ").strip()
        report_reply(data, user, reply_id, reason)


def search_menu(data):
    """Asks for search filters and shows the matching bottles."""
    keyword = input("Search word (press Enter to skip): ").strip()
    bottle_type = input("Type - advice, comfort or motivation (press Enter to skip): ").strip().lower()
    mood = input("Mood (press Enter to skip): ").strip().lower()
    results = search_bottles(data, keyword, bottle_type, mood)
    print("Found", len(results), "bottle(s)")
    for bottle in results:
        print(f"  Bottle {bottle['id']} ({bottle['type']}, {bottle['mood']}, {bottle['status']}): {bottle['message']}")


def leaderboard_menu(data):
    """Prints the Lighthouse leaderboard."""
    helpers = get_leaderboard(data)
    if len(helpers) == 0:
        print("Nobody has been marked helpful yet.")
        return
    rank = 1
    for user in helpers:
        if user["helped"] >= LIGHTHOUSE_LIMIT:
            print(f"{rank}. {user['alias']} - helpful replies: {user['helped']} - LIGHTHOUSE")
        else:
            print(f"{rank}. {user['alias']} - helpful replies: {user['helped']}")
        rank = rank + 1


def user_menu(data, user):
    """Menu for a logged in user. Keeps running until the user logs out."""
    catches_left = CATCH_LIMIT
    while True:
        print()
        print("=====", user["alias"], "=====")
        print("1. Throw a bottle")
        print("2. Catch a bottle (" + str(catches_left) + " left)")
        print("3. My bottles and replies")
        print("4. Search bottles")
        print("5. Lighthouse leaderboard")
        print("6. Generate report")
        print("7. Logout")
        choice = get_number("Choose: ", 1, 7)

        if choice == 7:
            write_log(user["alias"] + " logged out")
            print("Logged out. Take care!")
            break

        data["tide"] = data["tide"] + 1
        if choice == 1:
            throw_menu(data, user)
        elif choice == 2:
            catches_left = catch_menu(data, user, catches_left)
        elif choice == 3:
            my_bottles_menu(data, user)
        elif choice == 4:
            search_menu(data)
        elif choice == 5:
            leaderboard_menu(data)
        elif choice == 6:
            print(generate_report(data))
            print("Report saved in", REPORT_FILE)
        save_data(data, DATA_FILE)

### 11. Main function
`main()` starts the portal.
1. It loads the data from the file.
2. It shows the main menu in a `while True` loop until the user picks Exit.
3. If the user stops the program with Ctrl+C (or the stop button in Jupyter), `except KeyboardInterrupt` catches it.
4. `finally` always saves the data at the end, so nothing is lost.

In [11]:
def main():
    """Starts the portal: loads the data, shows the main menu and saves when finished."""
    data = load_data(DATA_FILE)
    print("==========================================")
    print("   ANONYMOUS SUPPORT AND ADVICE PORTAL")
    print("   Say it to the sea. Someone will answer.")
    print("==========================================")
    try:
        while True:
            print()
            print("1. Register")
            print("2. Login")
            print("3. Exit")
            choice = get_number("Choose: ", 1, 3)
            if choice == 1:
                username = input("Choose a username: ").strip()
                password = input("Choose a password: ")
                register_user(data, username, password)
                save_data(data, DATA_FILE)
            elif choice == 2:
                username = input("Username: ").strip()
                password = input("Password: ")
                user = user_login(data, username, password)
                if user != None:
                    user_menu(data, user)
            else:
                print("Goodbye! Take care.")
                break
    except KeyboardInterrupt:
        print("Portal stopped.")
    finally:
        save_data(data, DATA_FILE)
        print("All data saved.")

## Part 2: Testing our functions
In these cells we call our functions directly to check that they work, and also try wrong inputs to see that they are handled.

### Test 1: Loading the data
The first time, there is no data file, so the demo data is added.

In [12]:
data = load_data(DATA_FILE)
print("Users:", len(data["users"]))
print("Bottles:", len(data["bottles"]))

No data file found. Starting fresh with some demo data.
Account created! Other people will only see you as: Blue Tide 19
Account created! Other people will only see you as: Silver Anchor 72
Your bottle is drifting out to sea...
Your bottle is drifting out to sea...
Your bottle is drifting out to sea...
Users: 2
Bottles: 3


### Test 2: Registering (correct and wrong inputs)

In [13]:
register_user(data, "ali", "ali123")
register_user(data, "sara", "sara123")
register_user(data, "ali", "hello123")   # username taken
register_user(data, "", "hello123")      # empty username
register_user(data, "zain", "12")        # password too short

Account created! Other people will only see you as: Calm Tide 36
Account created! Other people will only see you as: Hidden Anchor 93
This username is already taken.
Username cannot be empty or have spaces.
Password must be at least 4 characters.


### Test 3: Login

In [14]:
ali = user_login(data, "ali", "ali123")
sara = user_login(data, "sara", "sara123")
wrong = user_login(data, "ali", "wrongpass")
print("Wrong login gives:", wrong)

Welcome back, Calm Tide 36
Welcome back, Hidden Anchor 93
Wrong username or password.
Wrong login gives: None


### Test 4: Throwing bottles

In [15]:
throw_bottle(data, ali, "motivation", "stressed", "My semester project is due and nothing is working.")
throw_bottle(data, ali, "money", "sad", "This type does not exist.")    # wrong type
throw_bottle(data, ali, "advice", "sad", "help")                        # too short
throw_bottle(data, sara, "comfort", "sad", "Some days I just want to die, everything feels too heavy.")

Your bottle is drifting out to sea...
Wrong bottle type or mood.
Message is too short, please write a bit more.
Your bottle is drifting out to sea...
It sounds like you are going through something very hard.
Please talk to someone you trust or call a mental health helpline.


### Test 5: Drifting, catching and replying
Ali's new bottle cannot be caught straight away. After the tide goes up by 3, it can be caught.

In [16]:
print("Tide:", data["tide"], "- bottles Sara can catch:", len(find_catchable(data, sara)))
data["tide"] = data["tide"] + 3
print("Tide:", data["tide"], "- bottles Sara can catch:", len(find_catchable(data, sara)))

Tide: 0 - bottles Sara can catch: 3
Tide: 3 - bottles Sara can catch: 4


In [17]:
bottle = catch_bottle(data, sara)
print("Sara caught bottle", bottle["id"], ":", bottle["message"])
reply_bottle(data, sara, bottle, "ok")   # too short
reply_bottle(data, sara, bottle, "You are not alone, take it one small step at a time.")
reply_bottle(data, sara, bottle, "Second reply to the same bottle")   # already replied

other = catch_bottle(data, sara)
print("Sara caught bottle", other["id"], "and throws it back")
throw_back(data, other)

Sara caught bottle 4 : My semester project is due and nothing is working.
Reply is too short, please write a bit more.
Your reply is floating back to them. Thank you for being kind!
Someone already replied to this bottle.
Sara caught bottle 2 and throws it back
You let the bottle drift away.


### Test 6: Helpful replies and reports
The owner of the bottle Sara answered checks their replies.

In [18]:
owner = find_by_id(data["users"], bottle["sender_id"])
print("The bottle belongs to", owner["alias"])
for my_bottle, replies in get_my_bottles(data, owner):
    print("Bottle", my_bottle["id"], "-", my_bottle["status"], "-", len(replies), "reply")

reply_id = len(data["replies"])
mark_helpful(data, owner, reply_id)
mark_helpful(data, owner, reply_id)    # marked already
mark_helpful(data, sara, reply_id)     # not Sara's bottle
mark_helpful(data, owner, 50)          # reply does not exist
report_reply(data, owner, reply_id, "")
report_reply(data, owner, reply_id, "just testing the report")

The bottle belongs to Calm Tide 36
Bottle 4 - replied - 1 reply
Thanks! We told Hidden Anchor 93 that their reply helped.
You already marked this reply as helpful.
You can only rate replies to your own bottles.
There is no reply with that number.
Please write a reason.
Thank you, the reply has been reported.


### Test 7: Searching

In [19]:
results = search_bottles(data, "exam", "", "")
print("Search 'exam':", len(results), "result")
results = search_bottles(data, "", "comfort", "")
print("Type comfort:", len(results), "results")
results = search_bottles(data, "", "advice", "confused")
print("Advice + confused:", len(results), "result")
results = search_bottles(data, "pizza", "", "")
print("Search 'pizza':", len(results), "results")

Search 'exam': 1 result
Type comfort: 2 results
Advice + confused: 1 result
Search 'pizza': 0 results


### Test 8: Leaderboard and report

In [20]:
leaderboard_menu(data)
print()
print(generate_report(data))

1. Hidden Anchor 93 - helpful replies: 1

========== PORTAL REPORT ==========
Users: 4
Bottles thrown: 5
Bottles answered: 1 (20%)
Replies written: 1
Helpful replies: 1
Reported replies: 1
Bottles by type:
   advice: 1
   comfort: 2
   motivation: 2
Most common mood: stressed
Top helpers:
   1. Hidden Anchor 93 (1 helpful)



### Test 9: Error handling
Wrong number inputs (`ValueError`), a damaged data file and a missing data file. The program keeps running in all cases.

In [21]:
print(check_number("abc", 1, 5))
print(check_number("9", 1, 5))
print(check_number("3", 1, 5))

Please enter a number, not letters.
None
Please enter a number from 1 to 5
None
3


In [22]:
import os

# make a broken file on purpose
file = open("broken.json", "w")
file.write("this is not json")
file.close()
broken = load_data("broken.json")
print("Users in broken file:", len(broken["users"]))
os.remove("broken.json")

missing = load_data("no_such_file.json")
print("Users after missing file:", len(missing["users"]))

Error: the data file is damaged. Starting with empty data.
Users in broken file: 0
No data file found. Starting fresh with some demo data.
Account created! Other people will only see you as: Quiet Gull 32
Account created! Other people will only see you as: Gentle Shell 84
Your bottle is drifting out to sea...
Your bottle is drifting out to sea...
Your bottle is drifting out to sea...
Users after missing file: 2


### Test 10: Saving and loading again
We save the data, load it again from the file and check that everything is still there.

In [23]:
save_data(data, DATA_FILE)
loaded = load_data(DATA_FILE)
print("Users:", len(loaded["users"]))
print("Bottles:", len(loaded["bottles"]))
print("Replies:", len(loaded["replies"]))
print()
print("First lines of the log file:")
file = open(LOG_FILE, "r")
lines = file.readlines()
file.close()
for line in lines[:10]:
    print(line.strip())

Data loaded from portal_data.json
Users: 4
Bottles: 5
Replies: 1

First lines of the log file:
New user joined as Blue Tide 19
New user joined as Silver Anchor 72
Blue Tide 19 threw bottle 1
Silver Anchor 72 threw bottle 2
Blue Tide 19 threw bottle 3
New user joined as Calm Tide 36
New user joined as Hidden Anchor 93
Calm Tide 36 logged in
Hidden Anchor 93 logged in
Calm Tide 36 threw bottle 4


## Part 3: Running the portal
Run this cell to use the portal. Demo accounts: **ayesha / 1234** and **bilal / 1234**. Pick 3 (Exit) when you are done.

In [ ]:
main()